<a href="https://colab.research.google.com/github/arunpremraj2007/Test_Bench/blob/main/LLM/Bert_QA_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Step 1: Install Necessary Libraries ---
!pip install --upgrade transformers datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.8 MB/s eta 0:00:00


In [2]:
# --- Step 2: Import Libraries ---
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

In [3]:
# --- Step 2: Load the Dataset and Model ---
print("Loading SQuAD dataset...")
# Use the full path "rajpurkar/squad" to bypass the URI parsing bug with "squad"
raw_datasets = load_dataset("rajpurkar/squad", split={
    "train": "train[:1%]",      # Use 1% of the training data
    "validation": "validation[:5%]" # Use 5% of the validation data
})

print("Loading DistilBERT model and tokenizer...")
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

Loading SQuAD dataset...


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:


# --- Step 3: Preprocess the Data ---
# This is the most complex step for QA. We need to tokenize the context and question
# together and then map the answer text to the token positions.
max_length = 384 # The maximum length of a feature (question and context)
doc_stride = 128 # The authorized overlap between two parts of the context when splitting

def preprocess_function(examples):
    # Tokenize the questions and contexts, allowing for truncation and overlap.
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second", # Truncate the context, not the question
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # The 'overflow_to_sample_mapping' maps each new feature back to its original example.
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    # 'offset_mapping' maps each token to its character position in the original text.
    offset_mapping = tokenized_examples.pop("offset_mapping")

    # Now we label our data with the start and end token positions.
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Grab the original example corresponding to this feature.
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # If no answers are given, set the cls_index as the answer.
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Start and end character index of the answer in the text.
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Find the start and end token indices.
            token_start_index = 0
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1

            token_end_index = len(offsets) - 1
            while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                token_end_index -= 1

            # If the answer is not fully inside the context, label it with cls_index.
            if not (token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                 tokenized_examples["start_positions"].append(cls_index)
                 tokenized_examples["end_positions"].append(cls_index)
            else:
                 tokenized_examples["start_positions"].append(token_start_index - 1) # Adjust for python slicing
                 tokenized_examples["end_positions"].append(token_end_index - 1)


    return tokenized_examples

print("Tokenizing the dataset...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, remove_columns=raw_datasets["train"].column_names)

Tokenizing the dataset...


Map:   0%|          | 0/876 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

In [5]:
# --- Step 4: Prepare Data for Training ---
# We can use the default data collator provided by Transformers for training.
from transformers import default_data_collator

train_dataset = tokenized_datasets["train"]
validation_dataset = tokenized_datasets["validation"]

In [ ]:
# --- Step 5: Configure and Fine-Tune the Model with Hugging Face Trainer ---
print("Setting up training arguments...")
# Use 'eval_strategy' instead of the deprecated 'evaluation_strategy'
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    use_cpu=not torch.cuda.is_available()  # Fallback to CPU if GPU is not available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer, # Changed from tokenizer to processing_class for compatibility with newer transformers versions
    data_collator=default_data_collator,
)

print("Starting fine-tuning...")
trainer.train()

Setting up training arguments...
Starting fine-tuning...


Epoch,Training Loss,Validation Loss


In [ ]:
# --- Step 6: Inference ---
import numpy as np

def ask_question(model, tokenizer, question, context):
    """Asks a question to the fine-tuned model based on a context."""
    # Tokenize the input
    inputs = tokenizer(question, context, return_tensors="pt")

    # Move inputs to the same device as the model
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    # Get the most likely start and end token positions
    start_index = torch.argmax(start_logits, dim=-1).item()
    end_index = torch.argmax(end_logits, dim=-1).item()

    # Decode the tokens between start and end to get the answer
    input_ids = inputs["input_ids"][0].tolist()
    answer_tokens = input_ids[start_index : end_index + 1]
    answer = tokenizer.decode(answer_tokens)

    return answer

# Example usage with a new context and question
new_context = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower."
new_question = "Who is the Eiffel Tower named after?"

print("\n--- Answering a New Question ---")
print(f"Context: {new_context}")
print(f"Question: {new_question}")
answer = ask_question(model, tokenizer, new_question, new_context)
print(f"Predicted Answer: {answer}")